# Phrases mal étiquetées (outliers)

Enregistrements annotés du jeu de données choisi dans la liste de la section « Jeu de données » (`canopy.datasets.vocalizations`, ses MFCC calculés s'ils manquent) : chants du canari marron1 ([M1-2016-spring](https://zenodo.org/records/6521932), `canary_marron1`, par défaut) ou du canari llb3 (`llb3`), découpés en phrases (cris et TRASH compris, SIL ignorés, étiquettes répétées fusionnées) ; syllabes du chant du moineau du Japon gy6or6 (`bengalese_gy6or6`), cris de baleines bleues et de rorquals à Kerguelen (`kerguelen2014`, écoutés accélérés 16 fois) et vocalisations ultrasonores de souris (`mouse_usv`, ralenties 16 fois ; une seule étiquette : pas d'outlier d'étiquette), leurs segments annotés tels quels :

1. **Phrases atypiques dans l'UMAP et le t-SNE** (annotations corrigées) : outliers dans l'UMAP et le t-SNE des MFCC des phrases, avec leur écoute, et chants aux annotations décalées ;
2. **Comparaison de détecteurs d'outliers** (marron1 : annotations du record Zenodo ; les autres : leurs annotations, sans évaluation) : kNN, critères UMAP/t-SNE, LOF et leurs intersections, évalués sur les phrases réétiquetées depuis le record, et écoute des outliers du détecteur choisi.

Les cellules d'écoute de la deuxième partie utilisent l'affichage des phrases et le tableau de la première (à exécuter d'abord). Les graphes 3D des phrases (ACP, UMAP, t-SNE) et la trajectoire d'un chant sont dans le notebook `1. Exploration`. Prérequis : les caches de MFCC (`canopy/datasets/marron1/boundary_features_*.pkl`, créés au premier chargement), l'audio, et pour la deuxième partie les CSV du record Zenodo (`ZENODO_DIR`).

## Jeu de données

Le jeu de données analysé par les cellules suivantes, choisi dans la liste (`canopy.datasets.vocalizations.DATASETS`) : son choix calcule et met en cache ses MFCC s'ils manquent (quelques secondes ; marron1 : l'audio téléchargé depuis Zenodo si besoin, puis environ une minute). Exécuter ensuite les cellules suivantes. La case `denoise` débruite l'audio écouté et son spectrogramme (cochée par défaut pour Kerguelen, prise en compte au clic suivant) : soustraction spectrale (gain de Wiener) du bruit de fond de chaque fréquence, mesuré dans les parties de l'enregistrement sans segment annoté (`canopy.datasets.vocalizations.denoised`).

In [ ]:
import ipywidgets as widgets
from IPython.display import display

from canopy.datasets.vocalizations import CANARY, DATASETS, mfcc_cached, prepare_mfcc

dataset_status = widgets.Output()


def prepare_dataset(change=None):
    """The MFCC of the chosen dataset, computed and cached if they are not."""
    with dataset_status:
        dataset_status.clear_output()
        name = dataset_choice.value
        denoise_choice.value = DATASETS[name].denoise  # (the default of the dataset)
        try:
            if not mfcc_cached(name):
                print(f"{name}: MFCC not cached, computing them ...", flush=True)
                prepare_mfcc(name)
            print(f"{name}: MFCC ready, run the next cells (the analysis of this dataset)")
        except FileNotFoundError as error:
            print(error)


# the dataset of the next cells (the previous choice when the cell is run again)
dataset_choice = widgets.Dropdown(options=list(DATASETS), value=globals().get("DATASET", CANARY), description="dataset",
                                  layout=widgets.Layout(width="320px"))
# denoising of the audio listened to and of its spectrogram (read at each click)
denoise_choice = widgets.Checkbox(value=DATASETS[dataset_choice.value].denoise, indent=False,
                                  description="denoise (noise profile of the parts without segment)")
dataset_choice.observe(prepare_dataset, names="value")
display(widgets.VBox([dataset_choice, denoise_choice, dataset_status]))
prepare_dataset()


## Phrases atypiques dans l'UMAP et le t-SNE (annotations corrigées)

### Chants et affichage des phrases

Le jeu de données choisi ci-dessus (`DATASET`, au plus `MAX_SONGS` enregistrements tirés au hasard ; `None` : tous) : les MFCC de ses enregistrements (sans plancher des dB, fenêtre et bande de fréquences propres à l'espèce, en cache au premier chargement) et leurs annotations, et l'affichage d'une phrase utilisé par les cellules d'écoute : la séquence des phrases de son chant (la phrase encadrée), le spectrogramme de la phrase et de ses `NEIGHBORS` voisines de chaque côté, leurs étiquettes en haut, et un lecteur (la phrase avec les silences qui l'entourent, jusqu'à `CONTEXT` secondes avant et après mais sans les phrases voisines, ou avec ses voisines). À exécuter d'abord (après le choix du jeu de données). Les graphes ACP, UMAP et t-SNE des phrases en 3D et la trajectoire d'un chant sont dans le notebook `1. Exploration`.

In [ ]:
import base64
import io

import anywidget
import ipywidgets as widgets
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import traitlets
from IPython.display import HTML, display
from matplotlib.colors import to_hex, to_rgb, to_rgba
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle
from matplotlib.ticker import FormatStrFormatter

from canopy.common import label_colors, phrase_colors as colors_of_phrases
from canopy.datasets.songs import song_phrases
from canopy.datasets.vocalizations import CANARY, denoised, load_vocalizations, noise_profile

if "dataset_choice" not in globals():
    raise NameError("Run the cell of the dataset first: it chooses the dataset (and computes its MFCC if needed)")
DATASET = dataset_choice.value  # the dataset chosen above (DATASETS)
MAX_SONGS = None  # recordings of the dataset, at most (drawn at random; None: all)
NEIGHBORS = 2  # phrases shown (and played in the second player) before and after the clicked one


# the recordings of the dataset (their MFCC without the floor of the dB), their annotations and their phrases
# (segments: SIL ignored, the repeated labels of the canary songs merged)
vocalizations = load_vocalizations(DATASET, MAX_SONGS)
mfcc_songs, mfcc_train_songs, mfcc_test_songs, mfcc_frame = (
    vocalizations[key] for key in ("songs", "train_songs", "test_songs", "frame"))
annotations, song_segments, settings = vocalizations["annotations"], vocalizations["phrases"], vocalizations["settings"]
split_of = {**{w: "train" for w in mfcc_train_songs}, **{w: "test" for w in mfcc_test_songs}} \
    if mfcc_test_songs else {}  # (the other datasets: no split)
CONTEXT = settings.context  # s of silence played before and after the clicked phrase, at most (and shown around)
SPECTROGRAM_N_FFT, SPECTROGRAM_HOP, SPECTROGRAM_FMAX = settings.spectrogram  # (and its highest frequency, Hz)
print(f"{DATASET}: {sum(map(len, song_segments.values()))} phrases of {len(mfcc_songs)} recordings, "
      f"{len(set(annotations['syll']) - set(settings.ignored_labels))} labels")

# colors of the phrases, the same in all the cells (TRASH: the dark grey of its segments in the first cells)
phrase_colors = {**(colors_of_phrases() if DATASET == CANARY else label_colors(
    sorted({label for phrases in song_segments.values() for _, _, label in phrases}))), "TRASH": "#555555"}

WIDTH = 1800  # px of the spectrograms and figures (a plotly widget does not follow the width of the notebook)

# display of a phrase (show_phrase): played (with the silences around it, up to CONTEXT seconds), and shown in the sequence of the phrases
# of its song and on the spectrogram of its NEIGHBORS phrases before and after (with a cursor following the playback)
wav_paths = vocalizations["wav_paths"]
noise_profiles = {}  # noise of each recording (in the parts without segment), computed at its first denoising


def listened(wave, audio):
    """The audio of an extract of the recording wave, denoised if the denoise case is checked (noise_profile of the
    whole recording, n_fft of the MFCC; unchanged without enough noise)."""
    if not globals().get("denoise_choice") or not denoise_choice.value:
        return audio
    n_fft = settings.mfcc["n_fft"]
    if wave not in noise_profiles:
        whole, rate = sf.read(wav_paths[wave], dtype="float32", always_2d=True)
        noise_profiles[wave] = noise_profile(whole.mean(axis=1), rate, song_segments[wave], n_fft)
    return audio if noise_profiles[wave] is None else denoised(audio, noise_profiles[wave], n_fft)

class SpectrogramPlayer(anywidget.AnyWidget):
    """Spectrogram image and its audio: a cursor follows the playback (its time in the song sent back in position), a
    click on the image plays from there, the buttons play the phrase (with its silences) or the whole
    extract; each new click on a point (clicks) plays its phrase. The audio is played at speed times its rate
    (infrasounds sped up, ultrasounds slowed down)."""
    _esm = """
    function render({ model, el }) {
      el.innerHTML = `
        <div style="position:relative;display:inline-block;max-width:100%">
          <img style="display:block;max-width:100%;cursor:pointer">
          <div class="cursor" style="position:absolute;top:0;bottom:0;width:2px;background:#4fc3f7;display:none;
                                      pointer-events:none"></div>
        </div>
        <div style="margin:4px 0;font:12px sans-serif;display:flex;align-items:center;gap:8px">
          <button class="phrase">&#9654; phrase</button><button class="all">&#9654; with neighbors</button>
          <audio controls style="height:32px"></audio>
          <span style="color:#777">(click on the spectrogram to play from there)</span>
        </div>`;
      const img = el.querySelector("img"), cursor = el.querySelector(".cursor"), audio = el.querySelector("audio");
      const [phraseButton, allButton] = [el.querySelector(".phrase"), el.querySelector(".all")];
      const [phraseText, allText] = model.get("buttons");  // (a single text: the second button hidden)
      phraseButton.textContent = "\u25B6 " + phraseText;
      allButton.textContent = "\u25B6 " + allText;
      if (allText === undefined) allButton.style.display = "none";
      let stopAt = null, stopTimer = null, sent = 0, pressed = false;
      // a button pressed elsewhere in the page (turning a 3D graph): no position sent until its release, plotly
      // putting back the camera of its layout at each update of a trace during the gesture
      const press = (e) => { pressed = !el.contains(e.target); }, release = () => { pressed = false; };
      document.addEventListener("pointerdown", press, true);
      for (const type of ["pointerup", "pointercancel"]) document.addEventListener(type, release, true);
      window.addEventListener("blur", release);
      const t0 = () => model.get("t0");
      const xOf = (t) => model.get("left")
        + (t - t0()) / (model.get("t1") - t0()) * (model.get("right") - model.get("left"));
      const tick = () => {
        const t = t0() + audio.currentTime * model.get("speed");
        cursor.style.left = (100 * xOf(t)) + "%";
        cursor.style.display = "block";
        if (!pressed && performance.now() - sent > 50) {  // (the time in the song, for Python: 20 per second at most)
          model.set("position", t);
          model.save_changes();
          sent = performance.now();
        }
        if (stopAt !== null && t >= stopAt) { audio.pause(); stopAt = null; }
        if (!audio.paused) requestAnimationFrame(tick);
      };
      const playFrom = (t, until) => {
        audio.currentTime = Math.max(t - t0(), 0) / model.get("speed");
        stopAt = until;
        clearTimeout(stopTimer);
        audio.play();
      };
      audio.addEventListener("play", () => requestAnimationFrame(tick));
      // the end of the played phrase: a timer (tick, at each frame of the display, stops up to a frame late: much
      // for a short segment), set when the audio plays or seeks, checked against the clock of the audio when it fires
      // (the audio may start later than its "playing" event: the timer is set again for what is left)
      const stopAtEnd = () => {
        clearTimeout(stopTimer);
        if (stopAt === null || audio.paused) return;
        const left = ((stopAt - t0()) / model.get("speed") - audio.currentTime) / audio.playbackRate;
        if (left <= 0.002) { audio.pause(); stopAt = null; return; }
        stopTimer = setTimeout(stopAtEnd, left * 1000);
      };
      for (const type of ["playing", "seeked"]) audio.addEventListener(type, stopAtEnd);
      audio.addEventListener("seeked", tick);
      img.addEventListener("click", (e) => {
        const r = img.getBoundingClientRect(), f = (e.clientX - r.left) / r.width;
        const t = t0() + (f - model.get("left")) / (model.get("right") - model.get("left")) * (model.get("t1") - t0());
        if (t >= t0() && t <= model.get("t1")) playFrom(t, null);
      });
      phraseButton.addEventListener("click", () => playFrom(...model.get("phrase")));
      allButton.addEventListener("click", () => playFrom(t0(), null));
      const load = () => {
        img.src = model.get("image");
        audio.src = model.get("audio");
        cursor.style.display = "none";
      };
      model.on("change:clicks", () => { load(); playFrom(...model.get("phrase")); });
      load();
      return () => {  // (the view removed)
        audio.pause();
        document.removeEventListener("pointerdown", press, true);
        for (const type of ["pointerup", "pointercancel"]) document.removeEventListener(type, release, true);
        window.removeEventListener("blur", release);
      };
    }
    export default { render };
    """
    image = traitlets.Unicode("").tag(sync=True)  # data URL of the PNG
    audio = traitlets.Unicode("").tag(sync=True)  # data URL of the WAV
    t0 = traitlets.Float(0).tag(sync=True)  # time in the song of the start and of the end of the audio (s)
    t1 = traitlets.Float(1).tag(sync=True)
    left = traitlets.Float(0).tag(sync=True)  # fractions of the width of the image at t0 and t1
    right = traitlets.Float(1).tag(sync=True)
    phrase = traitlets.List([0, 0]).tag(sync=True)  # the clicked phrase with its silences (start, end in the song)
    clicks = traitlets.Int(0).tag(sync=True)
    buttons = traitlets.List(["phrase", "with neighbors"]).tag(sync=True)  # texts of the buttons (one: the first only)
    position = traitlets.Float(-1).tag(sync=True)  # time in the song of the playback (from the browser)
    speed = traitlets.Float(1).tag(sync=True)  # playback speed (the audio written at speed times its rate)


def show_phrase(wave, phrases, j, header_output, phrase_player, title=""):
    """The phrase j of the song wave (phrases: those of the song): the sequence of the phrases of the song in the Output
    header_output, the spectrogram of the phrase and of its NEIGHBORS phrases before and after in the
    SpectrogramPlayer phrase_player, which plays the phrase with the silences around it (up to CONTEXT seconds, not
    reaching the neighboring phrases); title: a line above."""
    start, end, label = phrases[j]
    first, last = max(j - NEIGHBORS, 0), min(j + NEIGHBORS, len(phrases) - 1)  # phrases of the spectrogram
    rate = sf.info(wav_paths[wave]).samplerate
    t0 = max(phrases[first][0] - CONTEXT, 0)
    audio, _ = sf.read(wav_paths[wave], start=int(t0 * rate), stop=int((phrases[last][1] + CONTEXT) * rate),
                       dtype="float32", always_2d=True)
    audio = listened(wave, audio.mean(axis=1))
    # (each extract at the same loudness: the quiet recordings audible, e.g. at sea: peaks of 0.006)
    audio = 0.9 * audio / max(np.abs(audio).max(), 1e-12)
    t1 = t0 + len(audio) / rate

    # the phrases of the song as colored pills: the clicked one framed, its neighbors normal, the others faded
    def pill(text, color, k):
        r, g, b, _ = to_rgba(color)
        text_color = "#1a1a1a" if 0.299 * r + 0.587 * g + 0.114 * b > 0.6 else "white"  # (readable on the color)
        style = (f"background:{to_hex(color)};color:{text_color};"
                 "padding:2px 9px;border-radius:11px;font:12px sans-serif;"
                 + ("font-weight:bold;outline:2px solid #1a1a1a;outline-offset:1px;" if k == j else "")
                 + ("opacity:0.35;" if k is not None and not first <= k <= last else ""))
        return f'<span style="{style}">{text}</span>'
    sequence = " ".join([pill("START", "#d0d0d0", None)]
                        + [pill(lab, phrase_colors.get(lab, "grey"), k) for k, (_, _, lab) in enumerate(phrases)]
                        + [pill("END", "black", None)])

    with header_output:
        header_output.clear_output(wait=True)
        header = pill(label, phrase_colors.get(label, "grey"), j)
        title_line = f'<div style="font:bold 13px sans-serif;margin:8px 0 2px">{title}</div>' if title else ""
        display(HTML(f'{title_line}<div style="font:13px sans-serif;margin:6px 0">{header}'
                     f' phrase {j + 1} / {len(phrases)} of <b>{wave}</b>, {start:.2f}-{end:.2f} s ({end - start:.2f} s)'
                     f'{", denoised" if globals().get("denoise_choice") and denoise_choice.value else ""}'
                     f'</div><div style="line-height:2.1">{sequence}</div>'))

    # spectrogram of the phrase and of its neighbors, sent to the player with its audio
    fig_clip = Figure(figsize=(WIDTH / 100, 2.8))  # (a figure of its own, not pyplot's current one)
    ax = fig_clip.subplots()
    fig_clip.subplots_adjust(left=0.05, right=0.99, top=0.97, bottom=0.18)
    S_clip = librosa.amplitude_to_db(np.abs(librosa.stft(audio, n_fft=SPECTROGRAM_N_FFT,
                                                         hop_length=SPECTROGRAM_HOP)), ref=np.max)
    librosa.display.specshow(S_clip, sr=rate, hop_length=SPECTROGRAM_HOP, x_axis="time", y_axis="hz", ax=ax,
                             cmap="magma",
                             x_coords=t0 + np.arange(S_clip.shape[1]) * SPECTROGRAM_HOP / rate)
    ax.set_ylim(0, SPECTROGRAM_FMAX)
    for k in range(first, last + 1):  # the phrases on a band at the top, as in the first cell
        p_start, p_end, p_label = phrases[k]
        ax.axvspan(p_start, p_end, ymin=0.9, ymax=1, color=phrase_colors.get(p_label, "grey"))
        dark = np.dot(to_rgb(phrase_colors.get(p_label, "grey")), [0.299, 0.587, 0.114]) < 0.5  # (white label on it)
        ax.text((p_start + p_end) / 2, 0.95 * SPECTROGRAM_FMAX, p_label, ha="center", va="center", fontsize=9,
                fontweight="bold" if k == j else "normal",
                color="white" if dark else "black")
    ax.axvspan(t0, start, ymax=0.9, color="black", alpha=0.45)  # (the clicked phrase in full light)
    ax.axvspan(end, t1, ymax=0.9, color="black", alpha=0.45)
    ax.add_patch(Rectangle((start, 0), end - start, 0.9 * SPECTROGRAM_FMAX,
                           fill=False, edgecolor="white", linewidth=1.5))
    ax.set_xlim(t0, t1)
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.1f"))
    ax.set_xlabel("time in the song (s)")
    image = io.BytesIO()
    fig_clip.savefig(image, format="png", dpi=100)
    left, right = ax.get_position().x0, ax.get_position().x1  # (fractions of the width of the image at t0 and t1)
    sound = io.BytesIO()
    sf.write(sound, audio, round(rate * settings.speed), format="WAV", subtype="PCM_16")
    with phrase_player.hold_sync():
        phrase_player.image = "data:image/png;base64," + base64.b64encode(image.getvalue()).decode()
        phrase_player.audio = "data:audio/wav;base64," + base64.b64encode(sound.getvalue()).decode()
        phrase_player.t0, phrase_player.t1, phrase_player.left, phrase_player.right = t0, t1, left, right
        # (the silences around the phrase, up to CONTEXT seconds, the neighboring phrases excluded, but never a part
        # of the phrase when they overlap it)
        previous_end = min(phrases[j - 1][1], start) if j > 0 else t0
        next_start = max(phrases[j + 1][0], end) if j + 1 < len(phrases) else t1
        phrase_player.phrase = [max(start - CONTEXT, previous_end, t0), min(end + CONTEXT, next_start, t1)]
        phrase_player.speed = settings.speed
        phrase_player.clicks += 1  # (plays the phrase)
    phrase_player.layout.display = ""



### Phrases atypiques (outliers)

Phrases de tous les chants dont l'étiquette ne correspond pas à leur voisinage, avec des critères très stricts : leurs MFCC moyennés (comme dans les graphes 3D du notebook 1) réduits en 3D par UMAP et par t-SNE (mêmes réglages), une phrase est retenue si, **dans les deux espaces**, aucune de ses `K` plus proches voisines n'a son étiquette, la phrase de son étiquette la plus proche est plus loin que pour `PERCENTILE` % des phrases, et au moins `MIN_OTHER` de ses voisines ont une même autre étiquette (celle à laquelle elle ressemble).

Le premier tableau montre les chants dont les phrases ressemblent rarement à leur étiquette (moins de la moitié) : leurs annotations semblent décalées dans le temps, et le décalage qui accorde le mieux leurs étiquettes à un kNN des MFCC des autres chants est estimé. Le second liste les outliers (en gris, à la fin, ceux de ces chants) : un clic sur une ligne affiche et joue l'outlier (cellule précédente, à exécuter d'abord), le bouton ♪ d'une étiquette (la sienne ou celles auxquelles il ressemble) une phrase typique de cette étiquette, pour comparer (la suivante des `TYPICAL` plus typiques à chaque clic). La case de chaque outlier le valide (son étiquette est fausse) ; le bouton au-dessus du tableau copie la liste des outliers validés (chant, phrase, début, fin, étiquette, étiquettes auxquelles il ressemble), à coller ici pour corriger les annotations. Les cases cochées sont gardées quand la cellule est relancée. Dans les graphes, les outliers sont marqués (losanges : ceux des chants suspects), et un clic sur une phrase (outlier ou non) l'affiche et la joue sous le tableau. UMAP et t-SNE des 12 890 phrases : une minute environ.

In [ ]:
from collections import Counter

import anywidget
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import traitlets
import umap
from IPython.display import display
from matplotlib.colors import to_hex, to_rgb
from plotly.subplots import make_subplots
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler

K = 10  # neighbors of each phrase in the 3D spaces
MIN_OTHER = 8  # neighbors of a single other label, at least
PERCENTILE = 99  # nearest phrase of the same label further than for this percentage of the phrases
SHIFTS = np.round(np.arange(-4, 1.01, 0.05), 2)  # s, time shifts of the annotations tried for the suspicious songs
TYPICAL = 10  # typical phrases of each label, played in turn by the clicks on the label in the table

if "show_phrase" not in globals():
    raise NameError("Run the cell of the loading of the dataset first: it defines the display of a phrase")

# the phrases of all the songs (TRASH included): their mean MFCC (as in the 3D graphs of notebook 1), their UMAP
# and t-SNE (the same settings)
all_waves = mfcc_train_songs + mfcc_test_songs
all_phrases = {w: song_segments[w] for w in all_waves}
info = [(w, j) for w in all_waves for j in range(len(all_phrases[w]))]  # (song, phrase index) of each point


def mean_mfcc(wave, shift=0.0):
    """Mean MFCC of each phrase of the song, its annotation times shifted by shift seconds."""
    mfcc = mfcc_songs[wave]["mfcc"][:, :13]
    means = []
    for start, end, _ in all_phrases[wave]:
        f0 = int(np.clip(np.floor((start + shift) / mfcc_frame), 0, len(mfcc) - 1))
        f1 = int(np.clip(np.floor((end + shift) / mfcc_frame) + 1, f0 + 1, len(mfcc)))
        means.append(mfcc[f0:f1].mean(axis=0))
    return np.array(means)


all_scaler = StandardScaler()
X_all = all_scaler.fit_transform(np.vstack([mean_mfcc(w) for w in all_waves]))
all_labels = np.array([all_phrases[w][j][2] for w, j in info])
if len(np.unique(all_labels)) < 2:
    print(f"{DATASET}: a single label ({all_labels[0]}), no phrase can sound like another label: no outlier")
spaces = {"UMAP": umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=0,
                           n_jobs=1).fit_transform(X_all),
          "t-SNE": TSNE(n_components=3, perplexity=30, init="pca", random_state=0).fit_transform(X_all)}

# outliers: in each space, no phrase of their label among their K nearest neighbors, the nearest phrase of their label
# further than for PERCENTILE % of the phrases, and at least MIN_OTHER neighbors of a single other label (the label
# they sound like); kept when they are outliers in both spaces
flags, sounds_like = {}, {}
for name, Y in spaces.items():
    nearest = NearestNeighbors(n_neighbors=K + 1).fit(Y).kneighbors(Y)[1][:, 1:]  # (without the phrase itself)
    to_same = np.full(len(Y), np.inf)
    for label in np.unique(all_labels):
        idx = np.flatnonzero(all_labels == label)
        if len(idx) > 1:
            to_same[idx] = NearestNeighbors(n_neighbors=2).fit(Y[idx]).kneighbors(Y[idx])[0][:, 1]
    majority = [Counter(all_labels[n]).most_common(1)[0] for n in nearest]
    sounds_like[name] = np.array([label for label, _ in majority])
    flags[name] = ((all_labels[nearest] != all_labels[:, None]).all(axis=1)
                   & (to_same > np.percentile(to_same[np.isfinite(to_same)], PERCENTILE))
                   & np.array([count >= MIN_OTHER for _, count in majority]))
outlier = flags["UMAP"] & flags["t-SNE"]

# songs whose phrases rarely sound like their label (their neighbors in the UMAP): their annotations shifted in time?
# The shift maximizing the agreement with the labels of a kNN of the mean MFCC of the phrases of the other songs
agreement = pd.Series(sounds_like["UMAP"] == all_labels).groupby([w for w, _ in info]).mean()
outliers_per_song = pd.Series(outlier).groupby([w for w, _ in info]).sum()
suspicious = list(agreement[agreement < 0.5].index)
others = [w for w in all_waves if w not in suspicious]
knn = KNeighborsClassifier(n_neighbors=10).fit(
    all_scaler.transform(np.vstack([mean_mfcc(w) for w in others])),
    np.concatenate([[label for _, _, label in all_phrases[w]] for w in others]))
rows, best_shift = [], {}  # best_shift: estimated shift of the annotations of each suspicious song (s)
for w in suspicious:
    labels = np.array([label for _, _, label in all_phrases[w]])
    accuracy = [np.mean(knn.predict(all_scaler.transform(mean_mfcc(w, shift))) == labels) for shift in SHIFTS]
    best = int(np.argmax(accuracy))
    best_shift[w] = float(SHIFTS[best])
    rows.append({"song": w, "split": split_of.get(w, ""), "phrases": len(labels),
                 "sound like their label": agreement[w], "outliers": outliers_per_song[w],
                 "kNN agreement": accuracy[list(SHIFTS).index(0)], "best shift of the annotations (s)": SHIFTS[best],
                 "kNN agreement after the shift": accuracy[best]})
print(f"{outlier.sum()} outliers among {len(info)} phrases of {len(all_waves)} songs (UMAP only "
      f"{(flags['UMAP'] & ~outlier).sum()}, t-SNE only {(flags['t-SNE'] & ~outlier).sum()}); phrases sounding like "
      f"their label: {agreement.mean():.0%} per song on average, less than 50 % in {len(suspicious)} songs")
if rows:
    display(pd.DataFrame(rows).style.format(
        {"sound like their label": "{:.0%}", "kNN agreement": "{:.0%}", "kNN agreement after the shift": "{:.0%}",
         "best shift of the annotations (s)": "{:+.2f}"}, precision=2).hide(axis="index")
            .set_caption("Songs whose phrases rarely sound like their label (annotations shifted in time?)"))

# (the outliers of the suspicious songs last)
outlier_index = np.array(sorted(np.flatnonzero(outlier),
                                key=lambda k: (info[k][0] in suspicious, info[k][0], info[k][1])), dtype=int)
suspicious_point = np.isin([w for w, _ in info], suspicious)

# typical phrases of each label, to compare the outliers with: the phrases of the label nearest to the mean of its
# phrases in the space of the mean MFCC (outliers and phrases of the suspicious songs excepted)
typical = {}
for label in np.unique(all_labels):
    idx = np.flatnonzero(~outlier & ~suspicious_point & (all_labels == label))
    typical[label] = idx[np.argsort(np.linalg.norm(X_all[idx] - X_all[idx].mean(axis=0), axis=1))][:TYPICAL]
typical_turn = Counter()  # typical phrases of each label already played


class ClickableTable(anywidget.AnyWidget):
    """Table (scrolling, its header fixed) whose rows can be clicked, and the button of the cells which have one: each
    click sends back [row, column, "row" or "button", a number of its own] in clicked. With copy_lines (a line of text
    per row), a check box per row (the checked rows in checked) and a button copying copy_header and the lines of the
    checked rows (shown below it too, if the clipboard is not available)."""
    _esm = """
    function render({ model, el }) {
      const checkable = model.get("copy_lines").length > 0;
      el.innerHTML = `
        <style>.clickable-table tbody tr:hover td { filter: brightness(0.9); }</style>
        <div style="font:12px sans-serif;color:#777;margin:4px 0">${model.get("caption")}</div>
        <div style="font:12px sans-serif;margin:4px 0;align-items:center;gap:8px;
                    display:${checkable ? "flex" : "none"}">
          <button class="copy" style="font:12px sans-serif;cursor:pointer"></button>
          <span class="copied" style="color:#777"></span></div>
        <textarea class="copy-text" readonly style="display:none;width:900px;max-width:100%;height:110px;
                                                    font:11px monospace;margin:4px 0"></textarea>
        <div style="max-height:420px;overflow:auto;display:inline-block;border:1px solid #ddd">
          <table class="clickable-table" style="border-collapse:collapse;font:12px sans-serif">
            <thead></thead><tbody></tbody></table></div>`;
      const head = el.querySelector("thead"), body = el.querySelector("tbody");
      const th = (c) => `<th style="position:sticky;top:0;background:#f3f3f3;padding:4px 8px;text-align:left;
                                    border-bottom:1px solid #ccc;z-index:1">${c}</th>`;
      head.innerHTML = "<tr>" + (checkable ? th("validated") : "") + model.get("columns").map(th).join("") + "</tr>";
      const checked = new Set(model.get("checked"));
      const copyButton = el.querySelector(".copy"), copied = el.querySelector(".copied");
      const copyText = el.querySelector(".copy-text");
      const count = () => { copyButton.textContent = `\u29c9 copy the ${checked.size} validated rows`; };
      count();
      copyButton.addEventListener("click", async () => {
        const lines = [...checked].sort((a, b) => a - b).map((r) => model.get("copy_lines")[r]);
        copyText.value = [model.get("copy_header"), ...lines].join("\\n");
        copyText.style.display = "block";
        try {
          await navigator.clipboard.writeText(copyText.value);
          copied.textContent = `copied (${lines.length} rows)`;
        } catch (error) {  // (no clipboard: the selected text, to copy by hand if execCommand fails too)
          copyText.select();
          copied.textContent = document.execCommand("copy") ? `copied (${lines.length} rows)`
                                                            : "copy the text below";
        }
      });
      let selected = null;
      const send = (r, c, kind, element) => {
        if (selected) selected.style.outline = "";
        element.style.outline = "2px solid #e53935";
        element.style.outlineOffset = "-2px";
        selected = element;
        model.set("clicked", [r, c, kind, Date.now() + Math.random()]);  // (always a change, whatever the view)
        model.save_changes();
      };
      model.get("rows").forEach((row, r) => {
        const tr = body.insertRow();
        tr.style.cursor = "pointer";
        tr.title = model.get("row_title");
        if (checkable) {
          const td = tr.insertCell(), box = document.createElement("input");
          td.style.cssText = "padding:3px 8px;border-bottom:1px solid #eee;text-align:center";
          box.type = "checkbox";
          box.checked = checked.has(r);
          box.style.cursor = "pointer";
          box.addEventListener("click", (e) => e.stopPropagation());  // (not a click on the row)
          td.addEventListener("click", (e) => e.stopPropagation());
          box.addEventListener("change", () => {
            if (box.checked) checked.add(r); else checked.delete(r);
            model.set("checked", [...checked].sort((a, b) => a - b));
            model.save_changes();
            count();
          });
          td.appendChild(box);
        }
        row.forEach((cell, c) => {
          const td = tr.insertCell();
          td.textContent = cell.text;
          td.style.cssText = "padding:3px 8px;border-bottom:1px solid #eee;white-space:nowrap;" + (cell.style || "");
          td.addEventListener("click", () => send(r, c, "row", tr));
          if (cell.button) {
            const button = document.createElement("button");
            button.textContent = cell.button;
            button.title = cell.button_title || "";
            button.style.cssText = "margin-left:6px;font:11px sans-serif;padding:0 5px;cursor:pointer";
            button.addEventListener("click", (e) => { e.stopPropagation(); send(r, c, "button", button); });
            td.appendChild(button);
          }
        });
      });
    }
    export default { render };
    """
    caption = traitlets.Unicode("").tag(sync=True)
    columns = traitlets.List([]).tag(sync=True)
    rows = traitlets.List([]).tag(sync=True)  # rows of cells {text, style, button (its text), button_title}
    row_title = traitlets.Unicode("").tag(sync=True)  # tooltip of the rows
    clicked = traitlets.List([]).tag(sync=True)  # last click: [row, column, "row" or "button", a number of its own]
    copy_header = traitlets.Unicode("").tag(sync=True)  # first line of the copied text
    copy_lines = traitlets.List([]).tag(sync=True)  # line of each row in the copied text (none: no check boxes)
    checked = traitlets.List([]).tag(sync=True)  # checked rows


def label_cell(label):
    """Cell of a label, on its color, with a button playing a typical phrase of the label."""
    r, g, b = to_rgb(phrase_colors.get(label, "grey"))
    text_color = "#1a1a1a" if 0.299 * r + 0.587 * g + 0.114 * b > 0.6 else "white"  # (readable on the color)
    return {"text": str(label), "style": f"background:{to_hex((r, g, b))};color:{text_color};font-weight:bold",
            "button": "\u266a typical", "button_title": f"a typical {label} phrase (the next one at each click)"}


# validated outliers: (song, phrase index) of the checked rows, kept when the cell is run again
validated_outliers = globals().get("validated_outliers", set())
rows, copy_lines = [], []
for k in outlier_index:
    wave, j = info[k]
    start, end, label = all_phrases[wave][j]
    copy_lines.append("\t".join([wave, str(j + 1), f"{start:.3f}", f"{end:.3f}", label, sounds_like["UMAP"][k],
                                 sounds_like["t-SNE"][k]]))
    style = "color:#9a9a9a" if suspicious_point[k] else ""
    plain = [wave, split_of.get(wave, ""), j + 1, f"{start:.2f}", f"{end - start:.2f}"]
    rows.append([{"text": str(text), "style": style} for text in plain]
                + [label_cell(lab) for lab in (label, sounds_like["UMAP"][k], sounds_like["t-SNE"][k])]
                + [{"text": "yes" if suspicious_point[k] else "", "style": style}])
outlier_table = ClickableTable(
    columns=["song", "split", "phrase", "start (s)", "duration (s)", "label", "sounds like (UMAP)",
             "sounds like (t-SNE)", "suspicious song"],
    rows=rows, row_title="click: this outlier",
    caption=f"The {len(outlier_index)} outliers (in grey, at the end: those of the suspicious songs). Click a row to "
            "see and hear the outlier, the \u266a button of a label to see and hear a typical phrase of this label "
            f"(the next of its {TYPICAL} most typical ones at each click). Check the outliers whose label is wrong, "
            "and copy their list with the button.",
    copy_header="\t".join(["song", "phrase", "start (s)", "end (s)", "label", "sounds like (UMAP)",
                           "sounds like (t-SNE)"]),
    copy_lines=copy_lines, checked=[r for r, k in enumerate(outlier_index) if info[k] in validated_outliers])


def check_outliers(change):
    """The checked rows of the table: the validated outliers."""
    validated_outliers.difference_update(info[k] for k in outlier_index)
    validated_outliers.update(info[outlier_index[r]] for r in change["new"])


# the two spaces: the phrases faded, the outliers marked (diamonds: those of the suspicious songs); a click on a
# phrase (an outlier or not) shows and plays it below the table
fig_out = go.FigureWidget(make_subplots(rows=1, cols=len(spaces), specs=[[{"type": "scene"}] * len(spaces)],
                                        subplot_titles=list(spaces), horizontal_spacing=0.01))
for col, (name, Y) in enumerate(spaces.items(), start=1):
    for label in sorted(set(all_labels)):
        index = np.flatnonzero(all_labels == label)
        fig_out.add_trace(go.Scatter3d(
            x=Y[index, 0], y=Y[index, 1], z=Y[index, 2], mode="markers", name=label, legendgroup=label,
            showlegend=col == 1, customdata=index, hoverinfo="text",
            text=[f"{label} (sounds like {sounds_like[name][k]}), phrase {info[k][1] + 1} of {info[k][0]}"
                  for k in index],
            marker=dict(size=4 if label == "TRASH" else 2, color=to_hex(phrase_colors.get(label, "grey")),
                        opacity=0.8 if label == "TRASH" else 0.2)),  # (TRASH: a few points)
            row=1, col=col)
    fig_out.add_trace(go.Scatter3d(
        x=Y[outlier_index, 0], y=Y[outlier_index, 1], z=Y[outlier_index, 2], mode="markers", name="outliers",
        legendgroup="outliers", showlegend=col == 1, customdata=outlier_index, hoverinfo="text",
        text=[f"{all_labels[k]} (sounds like {sounds_like[name][k]}), phrase {info[k][1] + 1} of {info[k][0]}"
              for k in outlier_index],
        marker=dict(size=6, color=[to_hex(phrase_colors.get(all_labels[k], "grey")) for k in outlier_index],
                    symbol=["diamond" if info[k][0] in suspicious else "circle" for k in outlier_index],
                    line=dict(width=2, color="#e53935"), opacity=1)),
        row=1, col=col)
axis = dict(showticklabels=False, title="")
fig_out.update_scenes(xaxis=axis, yaxis=axis, zaxis=axis)
fig_out.update_layout(width=WIDTH, height=750, margin=dict(l=0, r=0, t=60, b=0),
                      legend=dict(itemsizing="constant", font_size=10),
                      title=f"The {outlier.sum()} outliers of the {len(info)} phrases of all the songs "
                            "(click a phrase to listen)")
outlier_header = widgets.Output()
outlier_player = SpectrogramPlayer(layout=widgets.Layout(display="none"))


def show_point(k, title=None):
    """The phrase k (index in info), shown and played below the table."""
    wave, j = info[k]
    if title is None:
        title = (f"{'Outlier' if outlier[k] else 'Phrase'} {all_labels[k]}, sounds like {sounds_like['UMAP'][k]} "
                 f"(UMAP) and {sounds_like['t-SNE'][k]} (t-SNE)"
                 + (" (suspicious song)" if suspicious_point[k] else ""))
    show_phrase(wave, all_phrases[wave], j, outlier_header, outlier_player, title)


def click_phrase(trace, points, selector):
    if points.point_inds:
        show_point(int(trace.customdata[points.point_inds[0]]))


def click_cell(change):
    """A click in the table: on a row, its outlier; on the button of a label, a typical phrase of this label (the next
    one at each click)."""
    row, column, kind, _ = change["new"]
    k = outlier_index[row]
    if kind == "row":
        show_point(k)
        return
    label = {5: all_labels[k], 6: sounds_like["UMAP"][k], 7: sounds_like["t-SNE"][k]}[column]
    if len(typical[label]):
        n = typical_turn[label] % len(typical[label])
        typical_turn[label] += 1
        show_point(typical[label][n], f"Typical {label} phrase ({n + 1} / {len(typical[label])}, the nearest to the "
                                      f"mean of the {label} phrases first)")


for trace in fig_out.data:
    trace.on_click(click_phrase)
outlier_table.observe(click_cell, names="clicked")
outlier_table.observe(check_outliers, names="checked")
display(widgets.VBox([outlier_table, outlier_header, outlier_player, fig_out]))

### Chants aux annotations décalées

Les chants suspects de la cellule précédente (à exécuter d'abord), entiers : leurs annotations (bande du haut) et ces annotations décalées (bande du dessous), du décalage estimé au départ, réglable avec le curseur. Le lecteur joue le chant, le curseur suit la lecture.

Les annotations des 5 chants trouvés décalés (172, 290, 365, 418, 430) sont corrigées dans les fichiers d'annotations (décalages estimés en alignant les segments annotés sur l'énergie du chant, voir `canopy.datasets.songs`) : ils ne sont plus suspects, mais restent dans la liste, avec leurs annotations corrigées (bande du haut) ; le curseur, à 0 au départ, les décale (à l'opposé de la correction : les annotations d'origine).

In [ ]:
import base64
import io

import ipywidgets as widgets
import librosa
import librosa.display
import numpy as np
import soundfile as sf
from IPython.display import display
from matplotlib.colors import to_rgb
from matplotlib.figure import Figure
from matplotlib.ticker import FormatStrFormatter

if "best_shift" not in globals():
    raise NameError("Run the cell of the outliers first: it estimates the shifts of the annotations of the suspicious "
                    "songs")

# s, shifts of the songs whose annotation files are corrected (canopy.datasets.songs): shown first,
# unshifted at first (the opposite shift: their original annotations); then the suspicious songs, at their estimated
# shift
CORRECTED = {"172_marron1_May_27_2016_38615584.wav": -0.810, "290_marron1_June_08_2016_25875641.wav": -0.990,
             "365_marron1_June_12_2016_43103084.wav": -1.715, "418_marron1_June_15_2016_49011949.wav": -1.980,
             "430_marron1_June_16_2016_28193589.wav": -1.380} if DATASET == CANARY else {}
first_shift = {**{w: 0.0 for w in CORRECTED}, **best_shift}

def show_shifted_song():
    """The spectrogram of the whole chosen song, its annotations at the top and the shifted ones below, and a player."""
    wave = shifted_song_choice.value
    audio, rate = sf.read(wav_paths[wave], dtype="float32", always_2d=True)
    audio = listened(wave, audio.mean(axis=1))
    # (each extract at the same loudness: the quiet recordings audible, e.g. at sea: peaks of 0.006)
    audio = 0.9 * audio / max(np.abs(audio).max(), 1e-12)
    t1 = len(audio) / rate
    fig_song = Figure(figsize=(WIDTH / 100, 3.4))  # (a figure of its own, not pyplot's current one)
    ax = fig_song.subplots()
    fig_song.subplots_adjust(left=0.05, right=0.99, top=0.9, bottom=0.15)
    S_song = librosa.amplitude_to_db(np.abs(librosa.stft(audio, n_fft=SPECTROGRAM_N_FFT,
                                                         hop_length=SPECTROGRAM_HOP)), ref=np.max)
    librosa.display.specshow(S_song, sr=rate, hop_length=SPECTROGRAM_HOP, x_axis="time", y_axis="hz", ax=ax,
                             cmap="magma")
    ax.set_ylim(0, SPECTROGRAM_FMAX)
    shifted = [(max(start + shift_slider.value, 0.0), max(end + shift_slider.value, 0.01), label)
               for start, end, label in all_phrases[wave]]  # (kept within the song)
    for phrases, low in ((all_phrases[wave], 0.9), (shifted, 0.8)):  # two bands of the phrases at the top
        for p_start, p_end, p_label in phrases:
            ax.axvspan(p_start, p_end, ymin=low, ymax=low + 0.1, color=phrase_colors.get(p_label, "grey"))
            if p_end - p_start > 0.008 * t1:  # (the label of the wide ones)
                dark = np.dot(to_rgb(phrase_colors.get(p_label, "grey")), [0.299, 0.587, 0.114]) < 0.5
                ax.text((p_start + p_end) / 2, (low + 0.05) * SPECTROGRAM_FMAX, p_label,
                        ha="center", va="center", fontsize=7,
                        color="white" if dark else "black")
    ax.set_xlim(0, t1)
    ax.xaxis.set_major_formatter(FormatStrFormatter("%.0f"))
    ax.set_xlabel("time in the song (s)")
    corrected = f" (corrected in the files by {CORRECTED[wave]:+.3f} s)" if wave in CORRECTED else ""
    ax.set_title(f"{wave}: annotations{corrected} (top band) and annotations shifted by {shift_slider.value:+.3f} s "
                 "(band below)", fontsize=10, loc="left")
    image, sound = io.BytesIO(), io.BytesIO()
    fig_song.savefig(image, format="png", dpi=100)
    sf.write(sound, audio, round(rate * settings.speed), format="WAV", subtype="PCM_16")
    shifted_song_view.children = [SpectrogramPlayer(
        image="data:image/png;base64," + base64.b64encode(image.getvalue()).decode(),
        audio="data:audio/wav;base64," + base64.b64encode(sound.getvalue()).decode(),
        t0=0, t1=t1, left=ax.get_position().x0, right=ax.get_position().x1, phrase=[0, t1], buttons=["song"],
        speed=settings.speed)]


def choose_shifted_song(change=None):
    """The chosen song, unshifted (corrected songs) or at its estimated shift."""
    if shift_slider.value != first_shift[shifted_song_choice.value]:
        shift_slider.value = first_shift[shifted_song_choice.value]  # (redraws)
    else:
        show_shifted_song()


if not suspicious:
    print("No suspicious song: the phrases of all the songs sound like their label"
          + (" (the corrected songs below)" if CORRECTED else ""))
if CORRECTED or suspicious:
    shifted_song_choice = widgets.Dropdown(
        options=[(f"{w} (corrected in the files by {s:+.3f} s)", w) for w, s in CORRECTED.items()]
                + [(f"{w} (suspicious, estimated shift {best_shift[w]:+.2f} s)", w) for w in suspicious],
        description="song", layout=widgets.Layout(width="560px"))
    shift_slider = widgets.FloatSlider(min=SHIFTS[0], max=max([SHIFTS[-1], *(-s for s in CORRECTED.values())]),
                                       step=0.005, description="shift (s)", continuous_update=False,
                                       readout_format="+.3f", layout=widgets.Layout(width="500px"))
    shifted_song_view = widgets.VBox()
    shifted_song_choice.observe(choose_shifted_song, names="value")
    shift_slider.observe(lambda change: show_shifted_song(), names="value")
    choose_shifted_song()
    display(widgets.VBox([widgets.HBox([shifted_song_choice, shift_slider]), shifted_song_view]))

## Comparaison de détecteurs d'outliers (annotations Zenodo)

Recherche des phrases dont l'étiquette ne correspond pas au son, par plusieurs détecteurs et leurs intersections, et leur écoute (mêmes graphes et même tableau que les outliers UMAP/t-SNE ci-dessus).

1. **Phrases et descripteurs** : les phrases de tous les chants (cris et TRASH compris, SIL ignorés, étiquettes répétées fusionnées), décrites par leurs 13 MFCC moyens (réduits en 3D par UMAP et t-SNE, comme dans la première partie) et par des descripteurs plus riches (moyenne et écart-type des 13 MFCC, moyenne de leurs deltas, log de la durée). Les chants dont les phrases ressemblent rarement à leur étiquette (moins de la moitié, dans l'UMAP) ont sans doute des annotations décalées dans le temps : leur décalage est estimé, et ils ne sont pas comptés dans l'évaluation.
2. **Détecteurs et évaluation** : kNN dans l'espace des descripteurs (les `k` plus proches phrases ont toutes une même autre étiquette), critères du première partie dans l'UMAP et le t-SNE, LOF par étiquette, et leurs intersections. Avec `ANNOTATIONS = "zenodo"`, les annotations du record Zenodo sont analysées et chaque détecteur est évalué sur les phrases réétiquetées depuis (les deux passes de corrections) : vrais positifs (phrases réétiquetées), faux positifs (les autres phrases signalées, vérifiées à l'écoute), toutes les paires de détecteurs et une validation croisée du choix de la paire (choisie sur la moitié des chants, évaluée sur l'autre).
3. **Écoute** : les phrases signalées par un détecteur (`DETECTOR` au départ, puis choisi dans une liste), toutes ou seulement les vrais ou les faux positifs, dans un tableau et dans les espaces UMAP et t-SNE (un clic sur une ligne ou un point montre et joue la phrase, le bouton ♪ d'une étiquette une phrase typique de cette étiquette, les cases valident les outliers et le bouton copie leur liste).

Prérequis : le cache des MFCC sans plancher (`canopy/datasets/marron1/boundary_features_44100_no_floor.pkl`), l'audio, et pour l'évaluation les CSV du record Zenodo (`ZENODO_DIR`). UMAP et t-SNE des ~13 000 phrases : une à deux minutes.

In [ ]:
import pathlib
from collections import Counter

import numpy as np
import pandas as pd
import umap
from IPython.display import display
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import StandardScaler

from canopy.datasets.songs import song_phrases

if "song_segments" not in globals():
    raise NameError("Run the cell of the loading of the dataset first: it loads the dataset (DATASET)")
# "zenodo" (marron1): those of the record (ZENODO_DIR), scored against the corrections; "corrected": those of
# the dataset (the other datasets: no evaluation)
ANNOTATIONS = "zenodo" if DATASET == CANARY else "corrected"
ZENODO_DIR = pathlib.Path.home() / "Code/Finebouche/heb-expansion/datasets/Canary/annotations/annotations"
MIN_SONG_AGREEMENT = 0.5  # songs whose phrases sound like their label less often: annotations shifted in time?
SHIFTS = np.round(np.arange(-4, 1.01, 0.05), 2)  # s, shifts of the annotations tried for these songs

# the phrases of all the songs (TRASH included, SIL ignored, repeated labels merged; those of the first cell)
all_waves = mfcc_train_songs + mfcc_test_songs
corrected_annotations = annotations
study_annotations = (pd.concat([pd.read_csv(f, index_col=0) for f in sorted(ZENODO_DIR.glob("*.csv"))])
               if ANNOTATIONS == "zenodo" else corrected_annotations)
study_phrases = ({w: song_phrases(study_annotations, w, ("SIL",)) for w in all_waves} if ANNOTATIONS == "zenodo"
                 else {w: song_segments[w] for w in all_waves})
study_info = [(w, j) for w in all_waves for j in range(len(study_phrases[w]))]  # (song, phrase index) of each phrase
song_of = np.array([w for w, _ in study_info])
study_labels = np.array([study_phrases[w][j][2] for w, j in study_info])


def phrase_features(wave, shift=0.0):
    """(13 mean MFCC, descriptors: mean and standard deviation of the 13 MFCC, mean of their deltas, log of the
    duration) of each phrase of the song, its annotation times shifted by shift seconds."""
    mfcc = mfcc_songs[wave]["mfcc"]  # (frames, 13 MFCC + deltas + delta-deltas)
    means, descriptors = [], []
    for start, end, _ in study_phrases[wave]:
        f0 = int(np.clip(np.floor((start + shift) / mfcc_frame), 0, len(mfcc) - 1))
        f1 = int(np.clip(np.floor((end + shift) / mfcc_frame) + 1, f0 + 1, len(mfcc)))
        x = mfcc[f0:f1]
        means.append(x[:, :13].mean(axis=0))
        descriptors.append(np.concatenate([x[:, :13].mean(axis=0), x[:, :13].std(axis=0), x[:, 13:26].mean(axis=0),
                                           [np.log(end - start)]]))
    return np.array(means), np.array(descriptors)


means, descriptors = (np.vstack(a) for a in zip(*[phrase_features(w) for w in all_waves]))
mean_scaler = StandardScaler().fit(means)
X_mean = mean_scaler.transform(means)  # (UMAP, t-SNE)
X_desc = StandardScaler().fit_transform(descriptors)  # (kNN, LOF)
study_spaces = {"UMAP": umap.UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=0,
                            n_jobs=1).fit_transform(X_mean),
          "t-SNE": TSNE(n_components=3, perplexity=30, init="pca", random_state=0).fit_transform(X_mean)}

# songs whose phrases rarely sound like their label (the label most of their 10 neighbors in the UMAP share), and the
# shift of their annotations agreeing best with a kNN of the mean MFCC of the phrases of the other songs
nearest_umap = NearestNeighbors(n_neighbors=11).fit(study_spaces["UMAP"]).kneighbors(study_spaces["UMAP"])[1][:, 1:]
umap_like = np.array([Counter(study_labels[n]).most_common(1)[0][0] for n in nearest_umap])
song_agreement = pd.Series(umap_like == study_labels).groupby(song_of).mean()
shifted_songs = list(song_agreement[song_agreement < MIN_SONG_AGREEMENT].index)
scored = ~np.isin(song_of, shifted_songs)  # (the phrases of the evaluation)
shift_knn = KNeighborsClassifier(n_neighbors=10).fit(X_mean[scored], study_labels[scored])
rows = []
for w in shifted_songs:
    labels = np.array([label for _, _, label in study_phrases[w]])
    accuracy = [np.mean(shift_knn.predict(mean_scaler.transform(phrase_features(w, shift)[0])) == labels) for shift in SHIFTS]
    best = int(np.argmax(accuracy))
    rows.append({"song": w, "phrases": len(labels), "sound like their label": song_agreement[w],
                 "kNN agreement": accuracy[list(SHIFTS).index(0)], "best shift (s)": SHIFTS[best],
                 "kNN agreement after the shift": accuracy[best]})
if rows:
    display(pd.DataFrame(rows).style.format(
        {"sound like their label": "{:.0%}", "kNN agreement": "{:.0%}", "kNN agreement after the shift": "{:.0%}",
         "best shift (s)": "{:+.2f}"}).hide(axis="index")
            .set_caption("Songs whose phrases rarely sound like their label (annotations shifted in time?)"))

# phrases relabeled since the record (ANNOTATIONS="zenodo"): less than half of their duration covered by phrases of
# their label in the corrected annotations (corrected_label: the label covering most of it)
corrected_label = np.full(len(study_info), "", dtype=object)
if ANNOTATIONS == "zenodo":
    corrected_phrases = {w: song_phrases(corrected_annotations, w, ("SIL",)) for w in all_waves}
    for k, (w, j) in enumerate(study_info):
        start, end, label = study_phrases[w][j]
        cover = Counter()
        for s, e, lab in corrected_phrases[w]:
            cover[lab] += max(min(e, end) - max(s, start), 0)
        if (sum(cover.values()) > 0 and cover[label] < 0.5 * (end - start)
                and cover.most_common(1)[0][0] != label):
            corrected_label[k] = cover.most_common(1)[0][0]
relabeled = (corrected_label != "") & scored
print(f"{len(study_info)} phrases of {len(all_waves)} songs ({ANNOTATIONS} annotations); {len(shifted_songs)} songs whose "
      f"annotations seem shifted (not scored)" + (f"; {relabeled.sum()} phrases relabeled since the record"
                                                  if ANNOTATIONS == "zenodo" else ""))

### Détecteurs et évaluation

Détecteurs de base, chacun signale des phrases :
- **kNN `k`** : les `k` phrases les plus proches dans l'espace des descripteurs ont toutes une même autre étiquette ;
- **UMAP `k` / t-SNE `k`** (`p99` : avec le percentile) : critères des outliers UMAP/t-SNE ci-dessus dans chaque espace, aucune des `k` voisines n'a son étiquette, au moins 80 % d'entre elles en ont une même autre, et (`p99`) la phrase de son étiquette la plus proche est plus loin que pour 99 % des phrases ;
- **LOF** : phrase atypique parmi celles de son étiquette (Local Outlier Factor, 20 voisines, sur les descripteurs).

`DETECTORS` : les détecteurs retenus (seuls ou intersections), `DETECTOR` celui affiché au départ dans l'écoute plus bas (puis choisi dans une liste). Avec `ANNOTATIONS = "zenodo"` : vrais et faux positifs de chaque détecteur (phrases des chants décalés exclues), toutes les paires de détecteurs de base (front de Pareto : le moins de faux positifs pour chaque nombre de vrais), et validation croisée du choix de la paire (choisie sur une moitié des chants, rappel d'au moins `CV_RECALL` ; évaluée sur l'autre moitié ; `CV_SPLITS` tirages), qui montre si les meilleures combinaisons sont sur-ajustées.

In [ ]:
from itertools import combinations

import plotly.graph_objects as go
from sklearn.neighbors import LocalOutlierFactor

CV_RECALL = 0.75  # recall (on half of the songs) of the pair chosen in the cross-validation, at least
CV_SPLITS = 100


def knn_rule(X, k):
    """Phrases whose k nearest phrases in X all share a single other label."""
    neighbor_labels = study_labels[NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X)[1][:, 1:]]
    return (neighbor_labels == neighbor_labels[:, :1]).all(axis=1) & (neighbor_labels[:, 0] != study_labels)


def space_rule(Y, k, percentile=None):
    """Criteria of the outliers of the first part in the space Y: none of the k nearest phrases has the label, at least 80 % of them share
    another one, and (percentile) the nearest phrase of the label is further than for percentile % of the phrases."""
    nearest = NearestNeighbors(n_neighbors=k + 1).fit(Y).kneighbors(Y)[1][:, 1:]
    neighbor_labels = study_labels[nearest]
    majority = np.array([Counter(row).most_common(1)[0][1] for row in neighbor_labels])
    flags = (neighbor_labels != study_labels[:, None]).all(axis=1) & (majority >= np.ceil(0.8 * k))
    if percentile is not None:
        to_same = np.full(len(Y), np.inf)
        for label in np.unique(study_labels):
            idx = np.flatnonzero(study_labels == label)
            if len(idx) > 1:
                to_same[idx] = NearestNeighbors(n_neighbors=2).fit(Y[idx]).kneighbors(Y[idx])[0][:, 1]
        flags &= to_same > np.percentile(to_same[np.isfinite(to_same)], percentile)
    return flags


base = {f"kNN {k}": knn_rule(X_desc, k) for k in (5, 10, 30, 50)}
for name, Y in study_spaces.items():
    for k, percentile in ((5, 99), (10, 99), (30, None)):
        base[f"{name} {k}" + (f" p{percentile}" if percentile else "")] = space_rule(Y, k, percentile)
base["LOF"] = np.zeros(len(study_info), bool)
for label in np.unique(study_labels):
    idx = np.flatnonzero(study_labels == label)
    if len(idx) > 25:
        base["LOF"][idx] = LocalOutlierFactor(n_neighbors=20).fit_predict(X_desc[idx]) == -1

DETECTORS = {
    "UMAP ∩ t-SNE (première partie)": base["UMAP 10 p99"] & base["t-SNE 10 p99"],
    "kNN 10": base["kNN 10"],
    "kNN 50": base["kNN 50"],
    "kNN 10 ∩ t-SNE": base["kNN 10"] & base["t-SNE 10 p99"],
    "kNN 50 ∩ UMAP": base["kNN 50"] & base["UMAP 5 p99"],
    "kNN 10 ∩ LOF": base["kNN 10"] & base["LOF"],
}
DETECTOR = "UMAP ∩ t-SNE (première partie)"  # shown first below (then chosen in the list)


def score(flags, mask=scored):
    flagged = flags & mask
    return dict(flagged=int(flagged.sum()), true=int((flagged & relabeled).sum()),
                false=int((flagged & ~relabeled).sum()))


summary = pd.DataFrame([{"detector": name, **score(flags)} for name, flags in DETECTORS.items()])
if ANNOTATIONS != "zenodo":
    display(summary[["detector", "flagged"]].style.hide(axis="index")
            .set_caption("Phrases flagged by each detector (songs with shifted annotations excepted)"))
else:
    summary["precision"] = summary["true"] / summary["flagged"].clip(lower=1)
    summary["recall"] = summary["true"] / relabeled.sum()
    display(summary.rename(columns={"true": "true positives", "false": "false positives"}).style.format(
        {"precision": "{:.0%}", "recall": "{:.0%}"}).hide(axis="index").set_caption(
        f"Each detector on the {scored.sum()} phrases of the songs without shifted annotations: true positives (of the "
        f"{relabeled.sum()} phrases relabeled since the record) and false positives"))

    # all the pairs of base detectors, their Pareto front, and the cross-validation of the choice of the pair
    combos = {name: flags for name, flags in base.items()}
    combos.update({f"{a} ∩ {b}": base[a] & base[b] for a, b in combinations(base, 2)})
    pairs = pd.DataFrame([{"combination": name, **score(flags)} for name, flags in combos.items()])
    front = pairs[pairs["true"] > 0].sort_values(["true", "false"]).groupby("true").head(1)
    front = front[front["false"] == front["false"][::-1].cummin()[::-1]]
    rng = np.random.default_rng(0)
    songs = np.unique(song_of[scored])
    names, flag_matrix = list(combos), np.array(list(combos.values())) & scored
    cv = []
    for _ in range(CV_SPLITS):
        half = np.isin(song_of, rng.choice(songs, len(songs) // 2, replace=False))
        true_a, false_a = (flag_matrix & half & relabeled).sum(1), (flag_matrix & half & ~relabeled).sum(1)
        ok = true_a >= CV_RECALL * (half & relabeled).sum()
        best = np.flatnonzero(ok)[np.argmin(false_a[ok])] if ok.any() else None
        other = scored & ~half
        for name, flags in [("best pair (chosen on the other half)", flag_matrix[best] if best is not None else None)] \
                + list(DETECTORS.items()):
            if flags is not None:
                cv.append(dict(detector=name, true=(flags & other & relabeled).sum(),
                               false=(flags & other & ~relabeled).sum(), n=(other & relabeled).sum()))
    cv = pd.DataFrame(cv).groupby("detector", sort=False)[["true", "false", "n"]].sum()
    cv = pd.DataFrame({"precision": cv["true"] / (cv["true"] + cv["false"]).clip(lower=1),
                       "recall": cv["true"] / cv["n"]})
    display(cv.style.format("{:.0%}").set_caption(
        f"Cross-validation ({CV_SPLITS} random halves of the songs): precision and recall on the other half"))

    fig_pairs = go.Figure()
    fig_pairs.add_trace(go.Scatter(x=pairs["true"], y=pairs["false"], mode="markers", name="all the pairs",
                                   text=pairs["combination"], hoverinfo="text+x+y",
                                   marker=dict(size=5, color="#b0b0b0")))
    fig_pairs.add_trace(go.Scatter(x=front["true"], y=front["false"], mode="lines+markers", name="Pareto front",
                                   text=front["combination"], hoverinfo="text+x+y", line=dict(color="#2a78d6")))
    fig_pairs.add_trace(go.Scatter(x=summary["true"], y=summary["false"], mode="markers+text", name="DETECTORS",
                                   text=summary["detector"], textposition="top left",
                                   marker=dict(size=11, symbol="diamond", color="#eb6834")))
    fig_pairs.update_layout(width=1000, height=560, xaxis_title=f"true positives (of {relabeled.sum()})",
                            yaxis_title="false positives", title="Detectors and pairs of base detectors")
    fig_pairs.show()

### Écoute des outliers

Les phrases signalées par le détecteur choisi dans la liste (parmi `DETECTORS` ; `DETECTOR` au départ, la méthode UMAP ∩ t-SNE de la première partie, avec ses faux positifs : `SUBSET`), toutes ou seulement, avec `ANNOTATIONS = "zenodo"`, les vrais positifs (réétiquetées depuis le record) ou les faux positifs (les autres, chants décalés exceptés). Le tableau donne l'étiquette à laquelle chaque phrase ressemble pour le kNN (étiquette commune à la plupart de ses 10 voisines dans l'espace des descripteurs) et dans l'UMAP, les détecteurs qui la signalent, son étiquette corrigée depuis le record et le verdict (vrai ou faux positif). Un clic sur une ligne montre et joue la phrase (la séquence des phrases de son chant et le spectrogramme de la phrase et de ses `NEIGHBORS` voisines), le bouton ♪ d'une étiquette une phrase typique de cette étiquette (la suivante des `TYPICAL` plus typiques à chaque clic). La case de chaque phrase la valide (son étiquette est fausse) ; le bouton au-dessus du tableau copie la liste des phrases validées. Dans les espaces UMAP et t-SNE, les phrases choisies sont marquées (losanges : celles des chants décalés ; contour vert : vrais positifs, rouge : faux positifs, gris : sans verdict), et un clic sur une phrase l'affiche et la joue sous le tableau.

In [ ]:
import ipywidgets as widgets
from matplotlib.colors import to_hex
from plotly.subplots import make_subplots

TYPICAL = 10  # typical phrases of each label, played in turn by the clicks on the label in the table
SUBSETS = ["all", "true positives", "false positives"] if ANNOTATIONS == "zenodo" else ["all"]
SUBSET = "false positives" if "false positives" in SUBSETS else "all"  # shown first (detector: DETECTOR), then chosen

if "show_phrase" not in globals() or "ClickableTable" not in globals():
    raise NameError("Run the cell of the loading of the dataset and the cell of the outliers (UMAP and t-SNE) first: they define the "
                    "display of a phrase and the table")
if DETECTOR not in DETECTORS:
    raise KeyError(f"DETECTOR {DETECTOR!r} is not one of DETECTORS: {list(DETECTORS)}")

# label most of the 10 nearest phrases share (descriptors), the detectors flagging each phrase, and its verdict
neighbor_labels = study_labels[NearestNeighbors(n_neighbors=11).fit(X_desc).kneighbors(X_desc)[1][:, 1:]]
knn_like = np.array([Counter(row).most_common(1)[0][0] for row in neighbor_labels])
flagged_by = [", ".join(name for name, f in DETECTORS.items() if f[k]) for k in range(len(study_info))]
shifted_point = ~scored
verdict = np.where(shifted_point | (ANNOTATIONS != "zenodo"), "",
                   np.where(relabeled, "true positive", "false positive"))
study_validated = globals().get("study_validated", set())  # (song, phrase index), kept when run again
study_typical_turn = Counter()


def select(detector, subset):
    """Phrases (indices in study_info) flagged by the detector, in the subset: "all" (those of the shifted songs last),
    "true positives" (relabeled since the record) or "false positives" (the others, shifted songs excepted)."""
    flags = DETECTORS[detector]
    if subset == "true positives":
        flags = flags & relabeled
    elif subset == "false positives":
        flags = flags & scored & ~relabeled
    return np.array(sorted(np.flatnonzero(flags), key=lambda k: (shifted_point[k], study_info[k])), dtype=int)


def typical_phrases(flags, n_typical=TYPICAL):
    """{label: its TYPICAL phrases nearest to the mean of its phrases (mean MFCC), the flagged phrases and those of the
    shifted songs excepted}."""
    typical = {}
    for label in np.unique(study_labels):
        idx = np.flatnonzero(~flags & scored & (study_labels == label))
        typical[label] = idx[np.argsort(np.linalg.norm(X_mean[idx] - X_mean[idx].mean(axis=0), axis=1))][:n_typical]
    return typical


def make_table(detector, subset, n_typical=TYPICAL):
    """The table of the phrases study_outliers, flagged by the detector (subset: its caption)."""
    verdict_style = {"true positive": "color:#2e7d32;font-weight:bold", "false positive": "color:#e53935;font-weight:bold"}
    rows, copy_lines = [], []
    for k in study_outliers:
        wave, j = study_info[k]
        start, end, label = study_phrases[wave][j]
        style = "color:#9a9a9a" if shifted_point[k] else ""
        plain = [wave, split_of.get(wave, ""), j + 1, f"{start:.2f}",
                 f"{end - start:.2f}"]
        rows.append([{"text": str(text), "style": style} for text in plain]
                    + [label_cell(lab) for lab in (label, knn_like[k], umap_like[k])]
                    + [{"text": flagged_by[k], "style": style},
                       label_cell(corrected_label[k]) if corrected_label[k] else {"text": "", "style": style},
                       {"text": "yes" if shifted_point[k] else "", "style": style},
                       {"text": verdict[k], "style": verdict_style.get(verdict[k], style)}])
        copy_lines.append("\t".join([wave, str(j + 1), f"{start:.3f}", f"{end:.3f}", label, knn_like[k], umap_like[k],
                                     corrected_label[k], verdict[k]]))
    what = {"all": "outliers", "true positives": "true positives (relabeled since the record)",
            "false positives": "false positives (not relabeled since the record)"}[subset]
    table = ClickableTable(
        columns=["song", "split", "phrase", "start (s)", "duration (s)", "label", "sounds like (kNN)",
                 "sounds like (UMAP)", "flagged by", "corrected since the record", "shifted song", "verdict"],
        rows=rows, row_title="click: this phrase",
        caption=f"The {len(study_outliers)} {what} of {detector}"
                + (" (in grey, at the end: those of the songs with shifted annotations)" if subset == "all" else "")
                + ". Click a row to see and hear the phrase, the ♪ button of a label to see and hear a typical "
                  f"phrase of this label (the next of its {n_typical} most typical ones at each click). Check the "
                  "phrases whose label is wrong, and copy their list with the button.",
        copy_header="\t".join(["song", "phrase", "start (s)", "end (s)", "label", "sounds like (kNN)",
                               "sounds like (UMAP)", "corrected since the record", "verdict"]),
        copy_lines=copy_lines, checked=[r for r, k in enumerate(study_outliers) if study_info[k] in study_validated])
    table.observe(click_study_cell, names="clicked")
    table.observe(check_study_outliers, names="checked")
    return table


def check_study_outliers(change):
    """The checked rows of the table: the validated outliers."""
    study_validated.difference_update(study_info[k] for k in study_outliers)
    study_validated.update(study_info[study_outliers[r]] for r in change["new"])


def show_study_point(k, title=None):
    """The phrase k (index in study_info), shown and played below the table."""
    wave, j = study_info[k]
    flagged = DETECTORS[detector_choice.value][k]
    if title is None:
        title = (f"{'Flagged phrase' if flagged else 'Phrase'} {study_labels[k]}, sounds like {knn_like[k]} "
                 f"(kNN) and {umap_like[k]} (UMAP)"
                 + (f", corrected to {corrected_label[k]}" if corrected_label[k] else "")
                 + (f" ({verdict[k]} of {detector_choice.value})" if flagged and verdict[k] else "")
                 + (" (shifted song)" if shifted_point[k] else ""))
    show_phrase(wave, study_phrases[wave], j, study_header, study_player, title)


def click_study_phrase(trace, points, selector):
    if points.point_inds:
        show_study_point(int(trace.customdata[points.point_inds[0]]))


def click_study_cell(change):
    """A click in the table: on a row, its phrase; on the button of a label, a typical phrase of this label (the next
    one at each click)."""
    row, column, kind, _ = change["new"]
    k = study_outliers[row]
    if kind == "row":
        show_study_point(k)
        return
    label = {5: study_labels[k], 6: knn_like[k], 7: umap_like[k], 9: corrected_label[k]}[column]
    if len(study_typical.get(label, [])):
        n = study_typical_turn[label] % len(study_typical[label])
        study_typical_turn[label] += 1
        show_study_point(study_typical[label][n], f"Typical {label} phrase ({n + 1} / {len(study_typical[label])}, the "
                                                  f"nearest to the mean of the {label} phrases first)")


# the two spaces: all the phrases faded, the selected ones marked (diamonds: those of the shifted songs; outline
# green: true positives, red: false positives, grey: no verdict); a click on a phrase shows and plays it below the table
study_fig = go.FigureWidget(make_subplots(rows=1, cols=len(study_spaces), specs=[[{"type": "scene"}] * len(study_spaces)],
                                          subplot_titles=list(study_spaces), horizontal_spacing=0.01))
marked_traces = []  # (index of the trace of the selected phrases, space) in each space
for col, (name, Y) in enumerate(study_spaces.items(), start=1):
    for label in sorted(set(study_labels)):
        index = np.flatnonzero(study_labels == label)
        study_fig.add_trace(go.Scatter3d(
            x=Y[index, 0], y=Y[index, 1], z=Y[index, 2], mode="markers", name=label, legendgroup=label,
            showlegend=col == 1, customdata=index, hoverinfo="text",
            text=[f"{label} (sounds like {knn_like[k]}), phrase {study_info[k][1] + 1} of {study_info[k][0]}"
                  for k in index],
            marker=dict(size=4 if label == "TRASH" else 2, color=to_hex(phrase_colors.get(label, "grey")),
                        opacity=0.8 if label == "TRASH" else 0.2)),
            row=1, col=col)
    study_fig.add_trace(go.Scatter3d(x=[], y=[], z=[], mode="markers", name="selected", legendgroup="selected",
                                     showlegend=col == 1, hoverinfo="text"), row=1, col=col)
    marked_traces.append((len(study_fig.data) - 1, Y))
axis = dict(showticklabels=False, title="")
study_fig.update_scenes(xaxis=axis, yaxis=axis, zaxis=axis)
study_fig.update_layout(width=WIDTH, height=750, margin=dict(l=0, r=0, t=60, b=0),
                        legend=dict(itemsizing="constant", font_size=10))
for trace in study_fig.data:
    trace.on_click(click_study_phrase)
study_header = widgets.Output()
study_player = SpectrogramPlayer(layout=widgets.Layout(display="none"))
detector_choice = widgets.Dropdown(options=list(DETECTORS), value=DETECTOR, description="detector",
                                   layout=widgets.Layout(width="420px"))
subset_choice = widgets.ToggleButtons(options=SUBSETS, value=SUBSET)
table_box = widgets.VBox()  # (only the table is rebuilt at each choice: the figure and the player stay)
study_table = None
study_box = widgets.VBox([widgets.HBox([detector_choice, subset_choice]), table_box, study_header, study_player,
                          study_fig])


def show_selection(change=None):
    """The phrases of the chosen detector and subset: in the table and marked in the two spaces."""
    global study_flags, study_outliers, study_typical, study_table
    detector, subset = detector_choice.value, subset_choice.value
    study_flags = DETECTORS[detector]
    study_outliers = select(detector, subset)
    study_typical = typical_phrases(study_flags)
    old_table, study_table = study_table, make_table(detector, subset)
    with study_fig.batch_update():
        for trace_index, Y in marked_traces:
            marked = study_fig.data[trace_index]
            marked.x, marked.y, marked.z = (Y[study_outliers, i] for i in range(3))
            marked.customdata = study_outliers
            marked.text = [f"{study_labels[k]} (sounds like {knn_like[k]})"
                           + (f", corrected to {corrected_label[k]}" if corrected_label[k] else "")
                           + (f", {verdict[k]}" if verdict[k] else "")
                           + f", phrase {study_info[k][1] + 1} of {study_info[k][0]}" for k in study_outliers]
            marked.marker = dict(
                size=6, opacity=1, color=[to_hex(phrase_colors.get(study_labels[k], "grey")) for k in study_outliers],
                symbol=["diamond" if shifted_point[k] else "circle" for k in study_outliers],
                line=dict(width=2, color=[{"true positive": "#2e7d32", "false positive": "#e53935"}.get(verdict[k], "#9a9a9a")
                                          for k in study_outliers]))
        study_fig.layout.title.text = (f"{len(study_outliers)} phrases ({subset}) of {detector} among the "
                                       f"{len(study_info)} phrases (click a phrase to listen)")
    table_box.children = [study_table]
    if old_table is not None:
        old_table.close()


detector_choice.observe(show_selection, names="value")
subset_choice.observe(show_selection, names="value")
show_selection()
display(study_box)